# Pre riesenie som pouzila Selenium z dovodu, ze Beautiful Soup hadzal chybu 403

Scrapping z notino.co.uk , ziskanie raw dat a ulozenie do csv:

In [9]:
import os
import logging
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
from time import sleep
import random

class NotinoScraper:
    def __init__(self, segment, country):
        self.segment = segment
        self.country = country
        self.base_url = f"https://www.notino.co.uk/{segment}/"
        self.driver = webdriver.Chrome()
        
        #Nastavenie loggingu pre scrapping:
        log_file_path = f"notino_scraper_{country}_log.log"
        logging.basicConfig(
            filename=log_file_path,
            filemode='a',
            format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
            level=logging.DEBUG,
            encoding='utf-8'
        )
        self.logger = logging.getLogger(__name__)
        self.logger.info(f'Initialized NotinoScraper for segment: {segment}, country: {country}')

    def clean_url(self, url):
        return url.strip(", ")

    def scrape_page(self, page_number):
        #scrapujem jednu stranku
        self.logger.info(f'Starting to scrape page {page_number}')
        products = []

        # Hledanie selectorov pomocou data-testid atributov v dOM
        try:
            product_elements = WebDriverWait(self.driver, 20).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '[data-testid="product-container"]'))
            )
            self.logger.debug(f"Found {len(product_elements)} product elements on page {page_number}")
        except Exception as e:
            self.logger.error(f"Error finding product elements on page {page_number}: {e}")
            return products

        for product in product_elements:
            try:
                name_element = product.find_elements(By.CSS_SELECTOR, 'img')
                price_element = product.find_elements(By.CSS_SELECTOR, '[data-testid="price-component"]')
                discount_element = product.find_elements(By.CSS_SELECTOR, 'span.styled__DiscountValue-sc-1b3ggfp-1')

                # Preskočenie bannerov:
                if not name_element or not price_element:
                    self.logger.debug(f"Skipping element with no name or price on page {page_number}")
                    continue

                name = name_element[0].get_attribute('alt').strip()
                price = price_element[0].text.strip()

                # Brand určím na základe prvého slova z name:
                brand = name.split()[0] if name else "N/A"

                image_url = name_element[0].get_attribute('src') if name_element else "N/A"
                product_url_element = product.find_elements(By.CSS_SELECTOR, 'a')
                product_url = product_url_element[0].get_attribute('href') if product_url_element else "N/A"
                product_url = self.clean_url(product_url)

                discount = discount_element[0].text.strip() if discount_element else "N/A"

                products.append({
                    'product_name': name,
                    'brand': brand,
                    'price': price,
                    'discount': discount,
                    'image_url': image_url,
                    'product_url': product_url
                })
            except Exception as e:
                self.logger.error(f'Error scraping product on page {page_number}: {e}')

        self.logger.info(f'Finished scraping page {page_number} with {len(products)} products')
        return products

    def scrape_all_pages(self):#scrapujem vsetky stranky
        all_products = []
        page_number = 1
        scraped_names = set()
        
        while True:
            self.logger.info(f"Scraping page {page_number}")
            print(f"Scraping page {page_number}")

            # Čakám na načítanie produktov na stránke, inak môže hodiť err
            WebDriverWait(self.driver, 30).until(
                EC.presence_of_all_elements_located((By.CSS_SELECTOR, '[data-testid="product-container"]'))
            )

            products = self.scrape_page(page_number)
            for product in products:
                if product['product_name'] not in scraped_names:
                    all_products.append(product)
                    scraped_names.add(product['product_name'])

            try:
                #Kliknutie na arrow button v pravom hornom rohu:
                next_button = WebDriverWait(self.driver, 30).until(
                    EC.element_to_be_clickable((By.CSS_SELECTOR, 'svg[data-testid="icon-regular-chevron-right"]'))
                )
                next_button.click()
                page_number += 1
                sleep(random.uniform(5, 8))  #Znova čakám pre prípad, že mi znova hodí err
            except Exception as e:
                self.logger.info(f"No more pages to scrape or an error occurred: {e}")
                print(f"No more pages to scrape or an error occurred: {e}")
                break

        self.driver.quit()

        #Výpis produktov:
        for product in all_products:
            print(f"Name: {product['product_name']}, Brand: {product['brand']}, Price: {product['price']}, Discount: {product['discount']}, URL: {product['product_url']} Image URL: {product['image_url']}")

        return pd.DataFrame(all_products)

    def scrape(self):
        self.logger.info(f"Starting scraping for {self.base_url}")
        self.driver.get(self.base_url)
        products_df = self.scrape_all_pages()
        self.logger.info("Scraping completed")
        return products_df

def main(segment: str, country: str) -> pd.DataFrame:
    scraper = NotinoScraper(segment=segment, country=country)
    products_df = scraper.scrape()
    products_df.to_csv(f"notino_raw_{country}.csv", index=False, encoding='utf-8')
    return products_df

# môžem si zvoliť segment a krajinu:
if __name__ == "__main__":
    df_raw = main(segment="toothpaste", country="uk")


Scraping page 1
Scraping page 2
Scraping page 3
Scraping page 4
Scraping page 5
Scraping page 6
Scraping page 7
Scraping page 8
Scraping page 9
Scraping page 10
Scraping page 11
Scraping page 12
Scraping page 13
Scraping page 14
Scraping page 15
Scraping page 16
Scraping page 17
Scraping page 18
Scraping page 19
Scraping page 20
Scraping page 21
No more pages to scrape or an error occurred: Message: 

Name: MEDIBLANC KIDS Raspberry toothpaste for children Rapsberry 50 ml, Brand: MEDIBLANC, Price: 6.60, Discount: -30 %, URL: https://www.notino.co.uk/mediblanc/kids-rapsberry-toothpaste-for-children/ Image URL: https://cdn.notinoimg.com/list_2k//mediblanc/8059300533123_01-o__220114.jpg
Name: Elmex Caries Protection Kids toothpaste for children, Brand: Elmex, Price: 2.90, Discount: N/A, URL: https://www.notino.co.uk/elmex/caries-protection-kids-toothpaste-for-children/ Image URL: https://cdn.notinoimg.com/list_2k//elmex/7610108056354_01-o__230215.jpg
Name: Elmex Caries Protection anti-deca

Transformacia povodneho csv, pridanie stplcov, kalkulacia slevy a final amount:

In [11]:

from datetime import datetime

def transform_data(input_file: str, output_file: str, country: str):
    df = pd.read_csv(input_file)

    # pridanie novych stplcov:
    df['country'] = country
    df['currency'] = '£'  # kedyž tak zmenit dle krajiny
    df['scraped_at'] = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

    # cleaning a pretypovanie
    df['price'] = df['price'].astype(str).str.replace('£', '').str.replace(',', '').astype(float)
    df['discount'] = df['discount'].astype(str).str.replace('%', '').str.replace(' ', '').astype(float)
    df['discount_amount'] = 0.0

    # ak je neake sleva, tak vykalkuluje final amount:
    df['final_price'] = df['price']
    for index, row in df.iterrows():
        if pd.notna(row['discount']) and row['discount'] != 0:
            discount_amount = row['price'] * (abs(float(row['discount'])) / 100)
            df.at[index, 'discount_amount'] = round(discount_amount, 2)
            df.at[index, 'final_price'] = round(row['price'] - discount_amount, 2)
        else:
            df.at[index, 'discount_amount'] = 0.0
            df.at[index, 'final_price'] = row['price']

# zaokruhlenie na 2 desatinne
    df['discount_amount'] = df['discount_amount'].round(2)
    df['final_price'] = df['final_price'].round(2)

    df.to_csv(output_file, index=False, encoding='utf-8-sig')
    print("Transformation successful.")
    
    #printujem df pre check
    print(df.head())
    
 #zakladni statistiky o cenach a slevach, min max je u discountu prohozene kvuli "-" znamenku
    print("Basic statistical information:")
    print(df[['price', 'discount', 'discount_amount']].describe())

if __name__ == "__main__":
    transform_data(input_file="notino_raw_uk.csv", output_file="notino_transformation.csv", country="uk")
    #check df 
    transformed_df = pd.read_csv("notino_transformation.csv")
    print("Transformed DataFrame:")
    print(transformed_df.head())
    print("Basic statistical information:")
    print(transformed_df[['price', 'discount', 'discount_amount']].describe())


Transformation successful.
                                        product_name      brand  price  \
0  MEDIBLANC KIDS Raspberry toothpaste for childr...  MEDIBLANC    6.6   
1  Elmex Caries Protection Kids toothpaste for ch...      Elmex    2.9   
2  Elmex Caries Protection anti-decay toothpaste ...      Elmex    3.9   
3  Colgate Natural Extracts Ultimate Fresh toothp...    Colgate    2.4   
4  Biorepair Plus Total Protection tooth enamel f...  Biorepair    3.8   

   discount                                          image_url  \
0     -30.0  https://cdn.notinoimg.com/list_2k//mediblanc/8...   
1       NaN  https://cdn.notinoimg.com/list_2k//elmex/76101...   
2       NaN  https://cdn.notinoimg.com/list_2k//elmex/40079...   
3       NaN  https://cdn.notinoimg.com/list_2k//colgate/871...   
4       NaN  https://cdn.notinoimg.com/list_2k//biorepair/b...   

                                         product_url country currency  \
0  https://www.notino.co.uk/mediblanc/kids-rapsbe...      